# Step 6: Fixed Lens Light, Free Mass + Source Light, Infer Flat $M/L$ + Spherical NFW

Workflow:
- use `/mnt/lustre/tianli/quasar_hmc/WFI2033_ss=2_full_light_20260326_14`
- read chain medians from `result/result_ss=2_full_light_20260326_14/HMC_median_draw_ss=2_full_light.nc`
- build one fixed lens light from:
  - inner 3 Gaussian from `fixed_first_three_gaussians`
  - outer 2 Gaussian from the full posterior median over `(chain, draw)`
- build one shared `data - lens light`
- keep lens light fixed
- free mass parameters and source light
- initialize each chain from its own chain median

Assumptions:
- perfect data
- `pixel_grid_shape = 75`
- point-source parameters stay fixed to each chain median
- stellar mass follows the fixed 5-Gaussian light profile with a flat `M/L`
- dark halo is spherical NFW with `R_s = 5 arcsec`


In [ ]:
import os
os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from pathlib import Path
from copy import deepcopy
import warnings
warnings.simplefilter('ignore')

import numpy as np
import xarray as xr
import jax
import jax.numpy as jnp
import numpyro
from numpyro import distributions as dist
from numpyro import infer
from numpyro.infer import autoguide, init_to_value
import optax
import matplotlib.pyplot as plt
from astropy.io import fits

from herculens_import_main import *
from lens_images_extension import LensImageExtension
from lens_images_extension import pixelize_plane as pixelize_plane_single
from herculens.PointSourceModel.point_source_model import PointSourceModel
from herculens.MassModel.mass_model import MassModel
from herculens.MassModel import mass_model_base
from herculens.LightModel.light_model import LightModel
from herculens.Instrument.psf import PSF
from herculens.Instrument.noise import Noise
from jax_lensing_profiles.MassModel.Profiles.CuspyNFW_ellipse_kappa import CuspyNFW_3D_fn
from jax_lensing_profiles.MassModel.Profiles.MGE import MGE

jax.config.update('jax_enable_x64', True)
numpyro.enable_x64()


In [ ]:
class CuspyNFWEllipseKappa(MGE):
    def __init__(self):
        super().__init__(
            CuspyNFW_3D_fn,
            'R_s',
            n_gauss=30,
            n_terms=28,
            sigma_start_mult=1e-4,
            sigma_end_mult=5,
            three_d=True,
        )

mass_model_base.STRING_MAPPING['CUSPY_NFW_ELLIPSE_KAPPA'] = CuspyNFWEllipseKappa


In [ ]:
suffix = '_ss=2_full_light'
RUN_OUTPUT_DIR = Path('/mnt/lustre/tianli/quasar_hmc/WFI2033_ss=2_full_light_20260326_14')
run_tag = '20260326_14'
products_root = Path('./result') / f'result{suffix}_{run_tag}'
products_dir = products_root / 'data_products'
NC_PATH = RUN_OUTPUT_DIR / f'WFI2033_all{suffix}.nc'
HMC_MEDIAN_PATH = products_root / f'HMC_median_draw{suffix}.nc'

DATA_DIR = Path('../../Data/WFI2033')
RAW_DATA_PATH = DATA_DIR / 'jw01198-o004_t004_nircam_clear-f115w_i2d.fits'
MASK_PATH = Path('data/mask_hmc.fits')
MASK_OUT_PATH = DATA_DIR / 'mask_out_center.fits'
FIXED_FIRST_THREE_PATH = RUN_OUTPUT_DIR / f'fixed_first_three_gaussians{suffix}.npz'
FIXED_FIRST_THREE_PSF_PATH = RUN_OUTPUT_DIR / f'fixed_first_three_psf{suffix}.fits'

with fits.open(RAW_DATA_PATH, memmap=True) as hdul_raw:
    raw_header = hdul_raw['SCI'].header if 'SCI' in hdul_raw else hdul_raw[0].header
    exposure_time = float(raw_header.get('EXPTIME', raw_header.get('TEXPTIME', raw_header.get('XPOSURE', 1.0))))
pix_scale = float(np.sqrt(raw_header['PIXAR_A2']))

data = np.array(fits.getdata(products_dir / f'data_bkg_sub{suffix}.fits'), dtype=float)
rms_file = np.array(fits.getdata(products_dir / f'rms_with_psf_extra{suffix}.fits'), dtype=float)
mask = np.array(fits.getdata(MASK_PATH), dtype=bool)
mask_out = np.array(fits.getdata(MASK_OUT_PATH), dtype=bool)
fixed_first_three = np.load(FIXED_FIRST_THREE_PATH)
fixed_first_three_psf = np.array(fits.getdata(FIXED_FIRST_THREE_PSF_PATH), dtype=float)

ny, nx = mask_out.shape
xc, yc = nx / 2, ny / 2
r = 16
Y, X = np.indices((ny, nx))
mask_out = np.logical_or(mask_out, (X - xc) ** 2 + (Y - yc) ** 2 <= r ** 2)
mask = np.array(mask_out, dtype=bool)

fixed_first_three_psf = np.clip(fixed_first_three_psf, 0.0, None)
fixed_first_three_psf /= fixed_first_three_psf.sum()

chain_median = xr.load_dataset(HMC_MEDIAN_PATH).isel(chain=slice(0, 4))
post_all = xr.open_dataset(NC_PATH, group='posterior', engine='h5netcdf')
lens_light_median = post_all[['amp_lens', 'sigma_lens', 'e_lens', 'center_lens', 'psf_kernel_corrected']].median(dim=('chain', 'draw')).load()
post_all.close()
num_chains = int(chain_median.sizes['chain'])

pixel_grid_shape = 75
source_grid_scale = 0.8
ss_factor = 2
k_values = K_grid((pixel_grid_shape, pixel_grid_shape)).k

SOURCE_GRID_PRIOR = {
    'plate_name': 'Source grid',
    'param_name': 'source_grid',
    'sigma_low': 1e-5,
    'sigma_high': 10.0,
    'n_high': 100,
    'positive': True,
}

G1_MASS_CENTER = (1.556, 1.299)
G2_MASS_CENTER = (2.145, -3.326)
conj_points = jnp.array([
    [1.20212170716053, -0.12271885209256231],
    [0.9053233071260114, 0.5277189685977776],
    [-1.0461673774453952, 1.0081083299749878],
    [-0.1255456241215261, -0.8965524340129204],
])

pixel_grid, xgrid, ygrid, x_axis, y_axis, extent, nx, ny = get_pixel_grid(jnp.asarray(data), pix_scale)
noise = Noise(nx, ny, exposure_time=exposure_time)
psf_global = np.array(lens_light_median['psf_kernel_corrected'].values, dtype=float)
psf_global = np.clip(psf_global, 0.0, None)
psf_global /= psf_global.sum()
psf = PSF(psf_type='PIXEL', kernel_point_source=psf_global)

print('RUN_OUTPUT_DIR =', RUN_OUTPUT_DIR)
print('NC_PATH =', NC_PATH)
print('HMC_MEDIAN_PATH =', HMC_MEDIAN_PATH)
print('products_dir =', products_dir)
print('num_chains =', num_chains)


In [ ]:
mass_model_step6 = MassModel([
    'MULTI_GAUSSIAN_ELLIPSE_KAPPA',
    'CUSPY_NFW_ELLIPSE_KAPPA',
    'SHEAR',
    'SIS',
    'SIS',
])

lens_light_model_dummy = LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {})
source_light_model = LightModel(
    ['PIXELATED'],
    pixel_adaptive_grid=True,
    pixel_interpol='fast_bilinear',
    kwargs_pixelated={'num_pixels': pixel_grid_shape},
)
point_source_model = PointSourceModel(
    ['IMAGE_POSITIONS'],
    mass_model=mass_model_step6,
    image_plane=deepcopy(pixel_grid),
)

lens_image_step6 = LensImageExtension(
    deepcopy(pixel_grid),
    deepcopy(psf),
    noise_class=noise,
    lens_light_model_class=lens_light_model_dummy,
    lens_mass_model_class=mass_model_step6,
    source_model_class=source_light_model,
    point_source_model_class=point_source_model,
    source_arc_mask=mask,
    conjugate_points=conj_points,
    kwargs_numerics={'supersampling_factor': ss_factor},
    source_grid_scale=source_grid_scale,
)

fixed_inner_three = [{
    'amp': np.array(fixed_first_three['amp'], dtype=float),
    'sigma': np.array(fixed_first_three['sigma'], dtype=float),
    'e1': np.array(fixed_first_three['e1'], dtype=float),
    'e2': np.array(fixed_first_three['e2'], dtype=float),
    'center_x': np.array(fixed_first_three['center_x'], dtype=float),
    'center_y': np.array(fixed_first_three['center_y'], dtype=float),
}]

outer_two_median = [{
    'amp': np.array(lens_light_median['amp_lens'].values, dtype=float)[-2:],
    'sigma': np.array(lens_light_median['sigma_lens'].values, dtype=float)[-2:],
    'e1': np.array(lens_light_median['e_lens'].values, dtype=float)[0, -2:],
    'e2': np.array(lens_light_median['e_lens'].values, dtype=float)[1, -2:],
    'center_x': np.array(lens_light_median['center_lens'].values, dtype=float)[0, -2:],
    'center_y': np.array(lens_light_median['center_lens'].values, dtype=float)[1, -2:],
}]

psf_outer_two = np.array(lens_light_median['psf_kernel_corrected'].values, dtype=float)
psf_outer_two = np.clip(psf_outer_two, 0.0, None)
psf_outer_two /= psf_outer_two.sum()


def concat_lens_light(inner_three, outer_two):
    return [{k: np.concatenate([np.asarray(inner_three[0][k]), np.asarray(outer_two[0][k])]) for k in ('amp', 'sigma', 'e1', 'e2', 'center_x', 'center_y')}]


def render_lens_light(kwargs_lens_light, psf_kernel):
    image_unconv = lens_image_step6.lens_surface_brightness(kwargs_lens_light)
    return np.array(lens_image_step6.ImageNumerics.re_size_convolve(image_unconv, unconvolved=False, psf_kernel=psf_kernel))


full_lens_light = concat_lens_light(fixed_inner_three, outer_two_median)
lens_light_image = render_lens_light(fixed_inner_three, fixed_first_three_psf) + render_lens_light(outer_two_median, psf_outer_two)
data_subtracted = np.array(data - lens_light_image, dtype=float)


def chain_array(name, i):
    return np.array(chain_median[name].isel(chain=i).values)


def build_chain_state(i):
    psf_corr = np.array(chain_array('psf_kernel_corrected', i), dtype=float)
    psf_corr = np.clip(psf_corr, 0.0, None)
    psf_corr = psf_corr / psf_corr.sum()
    return {
        'full_lens_light': full_lens_light,
        'data_subtracted': data_subtracted,
        'psf_kernel': psf_corr,
        'kwargs_point_source': [{
            'ra': chain_array('ra_ps', i),
            'dec': chain_array('dec_ps', i),
            'amp': np.power(10.0, chain_array('log10_amp_ps', i)),
        }],
        'init_values': {
            'm2l_ratio': jnp.array(1.0),
            'kappa_s_halo': jnp.array(0.1),
            'center_halo': jnp.asarray(chain_array('center_1', i)),
            'gamma_sheer_halo': jnp.asarray(chain_array('gamma_sheer_1', i)),
            'theta_E_g1': jnp.atleast_1d(jnp.asarray(chain_array('theta_E_g1', i))),
            'theta_E_g2': jnp.atleast_1d(jnp.asarray(chain_array('theta_E_g2', i))),
            'n_source_grid': jnp.asarray(chain_array('n_source_grid', i)),
            'rho_source_grid': jnp.asarray(chain_array('rho_source_grid', i)),
            'sigma_source_grid': jnp.asarray(chain_array('sigma_source_grid', i)),
            'pixels_wn_source_grid': jnp.asarray(chain_array('pixels_wn_source_grid', i)),
        },
    }


chain_states = [build_chain_state(i) for i in range(num_chains)]



In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
ax[0].imshow(np.ma.array(data, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
ax[0].set_title('data')
ax[1].imshow(np.ma.array(lens_light_image, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
ax[1].set_title('fixed lens light')
ax[2].imshow(np.ma.array(data_subtracted, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
ax[2].set_title('data - lens light')
plt.tight_layout()
plt.show()


In [ ]:
def build_mass_kwargs_from_params(light, params):
    amp = jnp.asarray(light['amp'])
    stellar = {
        'amp': amp / jnp.sum(amp) * params['m2l_ratio'],
        'sigma': jnp.asarray(light['sigma']),
        'e1': jnp.asarray(light['e1']),
        'e2': jnp.asarray(light['e2']),
        'center_x': jnp.asarray(light['center_x']),
        'center_y': jnp.asarray(light['center_y']),
    }
    halo_shear = params2kwargs_GNFW_w_shear(params, 'halo', Rs_value=5.0, gamma_in_value=1.0, sph=True)
    return [stellar] + halo_shear + params2kwargs_SIS(params, 'g1') + params2kwargs_SIS(params, 'g2')


def model_step6(data_subtracted, state):
    m2l_ratio = numpyro.sample('m2l_ratio', dist.Uniform(0.0, 14.0))
    halo_shear = GNFW_w_shear(
        'Halo',
        'halo',
        Rs_value=5.0,
        gamma_in_value=1.0,
        sph=True,
        gamma_sheer_low=-0.5,
        gamma_sheer_high=0.5,
    )
    sis_g1 = SIS('G1', 'g1', G1_MASS_CENTER, theta_low=0.0, theta_high=0.25)
    sis_g2 = SIS('G2', 'g2', G2_MASS_CENTER, theta_mean=0.622, theta_sigma=0.062)
    kwargs_source = [
        matern_power_spectrum(
            SOURCE_GRID_PRIOR['plate_name'],
            SOURCE_GRID_PRIOR['param_name'],
            k_values,
            n_high=SOURCE_GRID_PRIOR['n_high'],
            sigma_low=SOURCE_GRID_PRIOR['sigma_low'],
            sigma_high=SOURCE_GRID_PRIOR['sigma_high'],
            positive=SOURCE_GRID_PRIOR['positive'],
        )
    ]
    kwargs_lens = [
        {
            'amp': jnp.asarray(state['full_lens_light'][0]['amp']) / jnp.sum(jnp.asarray(state['full_lens_light'][0]['amp'])) * m2l_ratio,
            'sigma': jnp.asarray(state['full_lens_light'][0]['sigma']),
            'e1': jnp.asarray(state['full_lens_light'][0]['e1']),
            'e2': jnp.asarray(state['full_lens_light'][0]['e2']),
            'center_x': jnp.asarray(state['full_lens_light'][0]['center_x']),
            'center_y': jnp.asarray(state['full_lens_light'][0]['center_y']),
        }
    ] + halo_shear + sis_g1 + sis_g2

    model_image = lens_image_step6.model(
        kwargs_lens=kwargs_lens,
        kwargs_source=kwargs_source,
        kwargs_lens_light=[],
        kwargs_point_source=state['kwargs_point_source'],
        source_add=True,
        lens_light_add=False,
        point_source_add=True,
        psf_kernel=state['psf_kernel'],
    )
    numpyro.deterministic('model_image', model_image)

    with numpyro.plate(f'data - [{int(mask_out.sum())}]', int(mask_out.sum())):
        numpyro.sample('obs', dist.Normal(model_image[mask_out], jnp.asarray(rms_file)[mask_out]), obs=jnp.asarray(data_subtracted)[mask_out])


max_iterations = 5000
scheduler = optax.exponential_decay(init_value=5e-3, transition_steps=300, decay_rate=0.99)
optim = optax.adabelief(learning_rate=scheduler)
loss = infer.TraceMeanField_ELBO()

rng = jax.random.PRNGKey(1234)
keys = jax.random.split(rng, num_chains)
step6_results = []
step6_medians = []

for i, state in enumerate(chain_states):
    guide = autoguide.AutoDiagonalNormal(model_step6, init_loc_fn=init_to_value(values=state['init_values']), init_scale=0.01)
    svi = infer.SVI(model_step6, guide, optim, loss)
    result = svi.run(keys[i], max_iterations, state['data_subtracted'], state, progress_bar=True, stable_update=True)
    step6_results.append(result)
    step6_medians.append(guide.median(result.params))



In [ ]:
plt.figure(figsize=(10, 4))
for i, result in enumerate(step6_results):
    plt.plot(result.losses, alpha=0.5, label=f'chain {i}')
plt.yscale('log')
plt.xlabel('iteration')
plt.ylabel('ELBO loss')
plt.legend()
plt.show()


In [ ]:
def evaluate_step6_state(state, params, seed=0):
    trace = numpyro.handlers.trace(
        numpyro.handlers.substitute(
            numpyro.handlers.seed(model_step6, jax.random.PRNGKey(seed)),
            data=params,
        )
    ).get_trace(state['data_subtracted'], state)
    model_image = np.array(trace['model_image']['value'])
    params_full = dict(params)
    params_full['pixels_source_grid'] = trace['pixels_source_grid']['value']
    kwargs_all = {
        'kwargs_lens': build_mass_kwargs_from_params(state['full_lens_light'][0], params),
        'kwargs_source': [params2kwargs_power_spectrum(params_full, 'source_grid')],
        'kwargs_lens_light': [],
        'kwargs_point_source': state['kwargs_point_source'],
    }
    return model_image, kwargs_all


for i, (state, params) in enumerate(zip(chain_states, step6_medians)):
    model_image, kwargs_all = evaluate_step6_state(state, params, seed=100 + i)
    residual = (state['data_subtracted'] - model_image) / rms_file
    source, source_extent = pixelize_plane_single(lens_image_step6, kwargs_all, pixel_grid_shape, source_grid_scale=source_grid_scale)

    fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.suptitle(
        f'chain {i} | M/L = {float(np.asarray(params["m2l_ratio"]).reshape(-1)[0]):.3f} | '
        f'kappa_s = {float(np.asarray(params["kappa_s_halo"]).reshape(-1)[0]):.4f}',
        y=1.02,
    )

    ax[0].imshow(np.ma.array(state['data_subtracted'], mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
    ax[0].set_title('data - lens light')

    ax[1].imshow(np.ma.array(model_image, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
    ax[1].set_title('model')

    im = ax[2].imshow(np.ma.array(residual, mask=~mask_out), origin='lower', extent=extent, cmap='bwr', vmin=-3, vmax=3)
    ax[2].set_title('residual / rms')
    plt.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)

    ax[3].imshow(source, origin='lower', extent=source_extent, cmap='twilight')
    ax[3].set_title('source plane')

    plt.tight_layout()
    plt.show()

